In [41]:
import torch.nn as nn
import torch,copy
import torch.functional as F 

In [17]:
class LoraLinear(nn.Module):
    def __init__(
        self,
        base_layer: nn.Linear,      # 原来的线性层
        r: int = 8,                 # lora rank
        alpha: int = 16,            # lora alpha
        dropout_p: float = 0.0,     # lora dropout
        test_mode: bool = False,    # 测试模式，用于控制 lora_B 是否为全零
    ):
        super(LoraLinear, self).__init__()
        self.base_layer = copy.deepcopy(base_layer)
        self.r = r
        self.alpha = alpha
        self.dropout = nn.Dropout(dropout_p)

        # 定义 lora_A 和 lora_B 为 Parameter
        self.lora_A = nn.Parameter(torch.empty((r, base_layer.in_features), dtype=base_layer.weight.dtype))
        self.lora_B = nn.Parameter(torch.empty((base_layer.out_features, r), dtype=base_layer.weight.dtype))

        # 初始化 lora 矩阵
        nn.init.normal_(self.lora_A, mean=0.0, std=0.02)
        if test_mode:
            nn.init.normal_(self.lora_B, mean=0.0, std=0.02)
        else:
            nn.init.zeros_(self.lora_B)

        # 冻结原来的层的参数
        for param in self.base_layer.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scaling = float(self.alpha) / float(self.r)     # lora 缩放系数
        lora_adjustment = F.linear(self.dropout(x), self.lora_A)
        lora_adjustment = F.linear(lora_adjustment, self.lora_B)
        return self.base_layer(x) + lora_adjustment * scaling

- 递归查找，nn.Moduel的named_childer()会返回名称和下一级的子模块，注意这不会递归返回全部子模块，named_modules()才会返回所有子模块，所以需要递归向下查找替换
- 为什么不用named_modules()，这是因为不仅仅遍历，还需要替换参数，如果根据named_modules来遍历，会出现迭代中变更元素的错误，所以手动向下DFS
- 遇到nn.linear就替换

In [18]:
# 替换lora层
# 找到目标位置，替换


import torch.nn as nn

def replace_linear_with_lora(
    module: nn.Module,
    r: int = 8,
    alpha: int = 16,
    dropout_p: float = 0.0,
    embed_requires_grad: bool = False,      # embedding 层是否训练
    norm_requires_grad: bool = False,       # norm 层是否训练
    head_requires_grad: bool = False,       # lm_head 层是否训练（Causal LM才有）
    test_mode: bool = False,                # 测试模式，用于控制 lora_B 是否为全零
):
    """
    找到 module 中所有线性层并递归替换
    """
    for name, child in module.named_children():
        # 先处理额外的层，lm_head 也是 linear，所以先处理
        if any(s in name for s in ['embed', 'norm', 'lm_head']):
            requires_grad = embed_requires_grad if 'embed' in name \
                            else norm_requires_grad if 'norm' in name \
                            else head_requires_grad
            for param in child.parameters():
                param.requires_grad = requires_grad
        # 替换所有线性层，QLoRA 做法
        elif isinstance(child, nn.Linear):
            lora_linear = LoraLinear(child, r=r, alpha=alpha, dropout_p=dropout_p, test_mode=test_mode)
            setattr(module, name, lora_linear)
        # 递归向下替换
        else:
            replace_linear_with_lora(
                child, r, alpha, dropout_p,
                embed_requires_grad, norm_requires_grad, head_requires_grad,
                test_mode=test_mode
            )

In [5]:
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 30)
)

In [19]:
def print_trainable_parameters(model: nn.Module):
    """
    打印可训练参数，表现和 PeftModel 的 print_trainable_parameters 方法类似
    """
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    trainable_percentage = 100 * trainable_params / total_params
  
    # 返回可训练参数量、所有参数量、可训练参数量占比（百分比）
    print(f"trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {trainable_percentage:.4f}")

In [20]:
from transformers import AutoConfig

config = AutoConfig.for_model('llama')
config.hidden_size = 24
config.intermediate_size = config.hidden_size * 4
config.num_attention_heads = 4
config.num_hidden_layers = 4
config.num_key_value_heads = 2
config.vocab_size = 128

In [11]:
from transformers import AutoModel, AutoModelForCausalLM
raw_model = AutoModelForCausalLM.from_config(config)

In [21]:
raw_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128, 24)
    (layers): ModuleList(
      (0-3): 4 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=24, out_features=512, bias=False)
          (k_proj): Linear(in_features=24, out_features=256, bias=False)
          (v_proj): Linear(in_features=24, out_features=256, bias=False)
          (o_proj): Linear(in_features=512, out_features=24, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=24, out_features=96, bias=False)
          (up_proj): Linear(in_features=24, out_features=96, bias=False)
          (down_proj): Linear(in_features=96, out_features=24, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((24,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((24,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((24,), eps=1e-06)
    (rotary_emb): LlamaRotaryEmbedding()
  )
  (lm_h

In [22]:
print_trainable_parameters(raw_model)


trainable params: 181,464 || all params: 181,464 || trainable%: 100.0000


In [23]:
import copy
lora_model = copy.deepcopy(raw_model)  # 深克隆，独立一个新模型
replace_linear_with_lora(lora_model, r=8, alpha=16)  # 替换
print_trainable_parameters(lora_model) # 打印参数情况
# print(lora_model)

trainable params: 63,744 || all params: 245,208 || trainable%: 25.9959


In [24]:
lora_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128, 24)
    (layers): ModuleList(
      (0-3): 4 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LoraLinear(
            (base_layer): Linear(in_features=24, out_features=512, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (k_proj): LoraLinear(
            (base_layer): Linear(in_features=24, out_features=256, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (v_proj): LoraLinear(
            (base_layer): Linear(in_features=24, out_features=256, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (o_proj): LoraLinear(
            (base_layer): Linear(in_features=512, out_features=24, bias=False)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): LoraLinear(
            (base_layer): Linear(in_features=24

In [ ]:
# 检查是不是只有 LoRA 层是可训练的
def print_model_parameters(model):
    """
    查看模型参数的 requires_grad 情况
    """
    print("Layer Name & Parameters")
    print("----------------------------")
    for name, parameter in model.named_parameters():
        print(f"{name:50} | Requires_grad: {parameter.requires_grad}")

In [26]:
print_model_parameters(lora_model)

Layer Name & Parameters
----------------------------
model.embed_tokens.weight                          | Requires_grad: False
model.layers.0.self_attn.q_proj.lora_A             | Requires_grad: True
model.layers.0.self_attn.q_proj.lora_B             | Requires_grad: True
model.layers.0.self_attn.q_proj.base_layer.weight  | Requires_grad: False
model.layers.0.self_attn.k_proj.lora_A             | Requires_grad: True
model.layers.0.self_attn.k_proj.lora_B             | Requires_grad: True
model.layers.0.self_attn.k_proj.base_layer.weight  | Requires_grad: False
model.layers.0.self_attn.v_proj.lora_A             | Requires_grad: True
model.layers.0.self_attn.v_proj.lora_B             | Requires_grad: True
model.layers.0.self_attn.v_proj.base_layer.weight  | Requires_grad: False
model.layers.0.self_attn.o_proj.lora_A             | Requires_grad: True
model.layers.0.self_attn.o_proj.lora_B             | Requires_grad: True
model.layers.0.self_attn.o_proj.base_layer.weight  | Requires_grad:

In [27]:
# 和 hugging face 的 peft 对比
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules='all-linear', # 太低版本的 peft 不支持这种做法
)
peft_lora_model = copy.deepcopy(raw_model)
peft_lora_model = get_peft_model(peft_lora_model, lora_config)
peft_lora_model.print_trainable_parameters()


trainable params: 63,744 || all params: 245,208 || trainable%: 25.9959


In [29]:
print_model_parameters(peft_lora_model)

Layer Name & Parameters
----------------------------
base_model.model.model.embed_tokens.weight         | Requires_grad: False
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight | Requires_grad: False
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight | Requires_grad: True
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight | Requires_grad: True
base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight | Requires_grad: False
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight | Requires_grad: True
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight | Requires_grad: True
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight | Requires_grad: False
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight | Requires_grad: True
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight | Requires_grad: True
base_model.model.model.layers.0.self_attn.o_p

In [30]:
from typing import List

def unload_lora(module: nn.Module, adapter_name: str = 'adapter'):
    """
    卸载 lora 参数，并将原模型恢复至加载 lora 前的样子
    """
    lora_parameters = {}
    def search_lora_linear(module: nn.Module, prefix: List[str]):
        for name, child in module.named_children():
            new_prefix = prefix + [name]
            if isinstance(child, LoraLinear):
                # 保存 lora 参数
                lora_parameters['.'.join(new_prefix)] = {
                    "lora_A_weight": child.lora_A.data.cpu(),
                    "lora_B_weight": child.lora_B.data.cpu(),
                    "r": child.r,
                    "alpha": child.alpha,
                    "dropout_p": child.dropout.p,
                }
                setattr(module, name, child.base_layer)
            else:
                search_lora_linear(child, new_prefix)

    search_lora_linear(module, [])
    # 解冻原模型
    for name, param in module.named_parameters():
        param.requires_grad = True

    torch.save(lora_parameters, f"{adapter_name}.pt")

In [31]:
def load_lora(module: nn.Module, adapter_name: str = 'adapter'):
    """
    加载 lora 参数
    """
    lora_parameters = torch.load(f"{adapter_name}.pt")

    for name, lora_params in lora_parameters.items():
        child = dict(module.named_modules())[name]
        if isinstance(child, nn.Linear):
            lora_linear = LoraLinear(child, lora_params['r'], lora_params['alpha'], lora_params['dropout_p'])
            lora_linear.lora_A.data = lora_params["lora_A_weight"].to(lora_linear.lora_A.device)
            lora_linear.lora_B.data = lora_params["lora_B_weight"].to(lora_linear.lora_B.device)

            # 名称示例：layers.0.self_attn.q_proj
            # 根据名称循环找到所需 module
            parts = name.split(".")
            obj = module
            for part in parts[:-1]:  # 不包括最后一级
                obj = getattr(obj, part)
            setattr(obj, parts[-1], lora_linear)

    # 恢复原来的冻结方式，这里简单地除了 lora 全冻结
    for name, param in module.named_parameters():
        if any(s in name for s in ['embed', 'norm', 'lm_head']):
            param.requires_grad = False

In [32]:
bsz = 2
seq_len = 8
test_tensor = torch.randint(0, config.vocab_size, (bsz, seq_len))

In [33]:
lora_model = copy.deepcopy(raw_model)
replace_linear_with_lora(lora_model, r=8, alpha=16, test_mode=True)
# 开测试模式，让 BA 非零


In [36]:
# 原模型的前向结果
raw_model.eval()
print_trainable_parameters(raw_model)   # 检查参数和可训练情况
raw_res = raw_model(test_tensor)

trainable params: 181,464 || all params: 181,464 || trainable%: 100.0000


In [37]:
raw_res.__dict__

{'loss': None,
 'logits': tensor([[[ 0.1172,  0.0697, -0.1385,  ...,  0.0624,  0.0117,  0.1195],
          [ 0.0952,  0.0166, -0.2543,  ...,  0.0147,  0.0400,  0.0609],
          [ 0.1097,  0.0583, -0.2438,  ...,  0.0155,  0.0307,  0.0467],
          ...,
          [ 0.0109,  0.0213, -0.2552,  ...,  0.0625,  0.0411,  0.0278],
          [-0.0020,  0.0158, -0.3142,  ...,  0.0151,  0.0132,  0.0186],
          [ 0.0174,  0.0163, -0.2690,  ..., -0.0260,  0.0723,  0.0710]],
 
         [[-0.1323, -0.0233,  0.2081,  ..., -0.0775,  0.0018, -0.0997],
          [-0.1713, -0.1265,  0.1830,  ..., -0.0203,  0.0785, -0.0840],
          [-0.1210, -0.1004,  0.1583,  ..., -0.0322,  0.0673, -0.0359],
          ...,
          [-0.1148, -0.0834,  0.2315,  ..., -0.0464, -0.0004, -0.0475],
          [-0.1035, -0.1507,  0.1810,  ..., -0.0308,  0.0318, -0.0535],
          [-0.0921, -0.0981,  0.2018,  ..., -0.0755,  0.0508, -0.1172]]],
        grad_fn=<UnsafeViewBackward0>),
 'past_key_values': <transformers.ca

In [38]:
# 第一次直接初始化 lora 的前向结果
lora_model.eval()
print_trainable_parameters(lora_model) 

trainable params: 63,744 || all params: 245,208 || trainable%: 25.9959


In [39]:
# 卸载 lora 后的前向结果
unload_lora(lora_model)
lora_model.eval()
print_trainable_parameters(lora_model) 

trainable params: 181,464 || all params: 181,464 || trainable%: 100.0000


In [40]:
# 重新装载 lora 后的前向结果
load_lora(lora_model)
lora_model.eval()
print_trainable_parameters(lora_model)  # 检查参数和可训练情况

trainable params: 63,744 || all params: 245,208 || trainable%: 25.9959


In [ ]:
# 模型路径可以改成本地
model_name_or_path = 'Qwen/Qwen1.5-0.5B'

# 加载原始模型
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype=dtype).to(device)

In [ ]:
# 获取 lora model
replace_linear_with_lora(model, r=16, alpha=32, dropout_p=0.0)
model.to(device)

# 查看可训练参数
print_trainable_parameters(model)


# 实战

In [ ]:
# 数据路径可以改成本地
data_name_or_path = 'bio-nlp-umass/bioinstruct'

# 定义训练数据集
class SFTDataset(Dataset):
    def __init__(self,
        tokenizer: AutoTokenizer,
        data_path: str,
        load_local: bool = False,
        max_len: int = 256,
        split_len: str = '1%',
    ):
        super().__init__()
        self.tokenizer = tokenizer

        if load_local:
            ds = load_dataset('json', data_dir=data_path, split=f'train[:{split_len}]')
        else:
            ds = load_dataset(data_path, split=f'train[:{split_len}]')
        self.max_len = max_len

        def process_func(example):
            # 提取 instruction 和 input
            instruction = example['instruction'].strip()
            input = example['input'].strip()
            output = example['output'].strip()

            # 构造模板
            instruction_msg = [
                {"role": "user", "content": (instruction + f"\n{input}") if len(input) > 0 else instruction}
            ]
            tokenized_instruction = tokenizer.apply_chat_template(instruction_msg, tokenize=True, add_generation_prompt=True)
            tokenized_output = tokenizer(output + "<|im_end|>" + f"{tokenizer.eos_token}\n")['input_ids']

            # 截断，最大不超过 max_len
            tokenized_prompt = (tokenized_instruction + tokenized_output)[:self.max_len]

            # 构造 input_ids, attention_mask, labels
            input_ids = tokenized_prompt[:-1]
            padding_mask = ([0] * len(tokenized_instruction) + [1] * (len(tokenized_output)))[:self.max_len][1:]
            labels = tokenized_prompt[1:]

            return {
                'input_ids': input_ids,
                'attention_mask': padding_mask,
                'labels': labels,
            }

        self.ds = ds.map(
            process_func,
            batched=False,
            remove_columns=ds.column_names,
            desc='Processing dataset',
        )

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, index: int):
        return self.ds[index]

ChatML 格式

<|im_start|>system
You are a helpful assistant<|im_end|>
<|im_start|>user
{指令}<|im_end|>
<|im_start|>assistant
{回复}<|im_end|>

要注意：我们的数据集是单轮问答 QA 形式，不同于预训练对所有位置都计算 loss，在做 SFT 的时候，我们只计算 Answer 部分的 loss，Question 部分我们会 mask 掉。

In [ ]:
ds = SFTDataset(tokenizer, data_name_or_path, load_local=False, split_len="1%")

print(len(ds[0]['input_ids']))
print(len(ds[0]['attention_mask']))
print(len(ds[0]['labels']))

print(tokenizer.decode(ds[0]['input_ids']))
print(ds[0]['attention_mask'])
print(tokenizer.decode(ds[0]['labels']))

In [ ]:
def collate_fn(batch: List, tokenizer):
    max_len = max(len(item['input_ids']) for item in batch)

    input_ids = []
    attention_mask = []
    labels = []

    for item in batch:
        input_id = item['input_ids']
        attention_mask_item = item['attention_mask']
        label = item['labels']

        # 计算填充长度
        pad_len = max_len - len(input_id)

        # 左填充
        input_ids.append([tokenizer.eos_token_id] * pad_len + input_id)
        attention_mask.append([0] * pad_len + attention_mask_item)
        labels.append([tokenizer.eos_token_id] * pad_len + label)

    # 将 list 转换为 tensor
    input_ids = torch.LongTensor(input_ids)
    attention_mask = torch.LongTensor(attention_mask)
    labels = torch.LongTensor(labels)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }